# AllDap 채점 모델 파인튜닝: 채점 (학습 전 M3, 학습 후 M3)

설계: `docs/superpowers/specs/2026-10-05-judge-finetune-design.md` (8-1절)

**실행 방법**
1. 런타임 > 런타임 유형 변경 > **T4 GPU**. 왼쪽 열쇠 아이콘(보안 비밀)에 이름 `HF_TOKEN` 으로 Hugging Face 토큰을 넣는다.
2. Drive 의 `MyDrive/alldap_judgeft/` 에 `cases_new.jsonl`(새 시험지 `testdata/judgeft/cases.jsonl`)과 `cases_frozen.jsonl`(동결 시험지 `testdata/judge5v2/cases.jsonl`)을 올린다. 학습 후 M3 를 돌릴 때는 학습 노트북이 만든 `adapter/` 가 같은 폴더에 있어야 한다.
3. 둘째 칸의 `SHEET`, `MODEL`, `RUN` 을 오케스트레이터가 알려 준 값으로 바꾼다. 학습 후 M3 이면 오케스트레이터가 준 `ADAPTER_SHA256 = '...'` 한 줄을 그 자리에 그대로 붙여 넣는다. 학습 전 M3 이면 `ADAPTER_SHA256` 을 비워 둔다.
4. **모두 실행**. 끊기면 세션을 다시 시작하고 같은 값으로 모두 실행한다. 이미 채점한 문항은 건너뛴다(반쯤 쓴 마지막 줄은 버리고 그 문항부터 다시 한다).
5. 끝나면 마지막 칸이 결과 파일 둘(`<모델>_run<번호>.jsonl`, `.env.json`)을 담은 zip 을 내려받는다.

**학습이 끝날 때까지 새 시험지의 결과 파일을 열어 보지 않는다**(스펙 6-5절). 내려받은 zip 은 그대로 오케스트레이터에 넘긴다.
생성 설정은 v2 와 같다(탐욕적 디코딩, 최대 출력 512 토큰, repetition_penalty 1.0).

**처음 여는 법:** colab.research.google.com 에서 파일 > 노트북 업로드 로 이 `.ipynb` 파일을 고른다.

**허용 창:** 첫 실행에서 두 창이 뜬다. "이 노트북에 보안 비밀 액세스 권한을 부여하시겠습니까?" 에는 **액세스 권한 부여**, "이 노트북에서 Google Drive 파일에 액세스하도록 허용하시겠습니까?" 에는 **Google Drive에 연결** 을 누르고 계정을 고른 뒤 모두 허용한다. `HF_TOKEN` 이 목록에 있으면 그 줄의 "노트북 액세스" 스위치를 켠다.

**오류가 나면:** 모두 실행이 그 칸에서 멈추고 zip 도 내려받아지지 않는다. 빨간 오류 글의 **마지막 다섯 줄쯤** 을 복사해 그대로 알려 준다. 칸을 고치거나 값을 바꾸지 않는다.

**오래 걸리는 동안:** 브라우저 탭을 닫지 않고 열어 둔다. 컴퓨터의 잠자기를 끈다(맥: 시스템 설정 > 디스플레이 또는 배터리 > 잠자기 방지). 무료 GPU 한도에 걸려 "GPU 백엔드에 연결할 수 없습니다" 가 뜨면 몇 시간 뒤(보통 다음 날) 다시 연다. 이미 한 일은 Drive 에 남아 이어 한다.

In [ ]:
!pip install -q "transformers==4.57.6" "bitsandbytes==0.50.2" "peft==0.17.1" "accelerate==1.10.1" huggingface_hub


In [ ]:
SHEET = 'frozen'      # 'new' 또는 'frozen'
MODEL = 'M3pre'       # 'M3pre'(학습 전) 또는 'M3ft'(학습 후)
RUN = 1               # 1, 2, 3
ADAPTER_SHA256 = ''   # MODEL 이 'M3ft' 일 때 학습 노트북의 어댑터 해시
import gc, glob, hashlib, json, math, os, platform, random, shutil, time
import torch, transformers, bitsandbytes, peft, accelerate
from google.colab import drive, userdata, files
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig

drive.mount('/content/drive')
DIR = '/content/drive/MyDrive/alldap_judgeft'
HF_TOKEN = userdata.get('HF_TOKEN')
M3_REPO = 'Unbabel/M-Prometheus-14B'
M3_REVISION = 'f790ee744ba08731697e9347c6ab709f9405a37e'   # v2 와 같은 리비전으로 고정한다(스펙 6-1절)
PINS = {'transformers': '4.57.6', 'bitsandbytes': '0.50.2', 'peft': '0.17.1', 'accelerate': '1.10.1'}

def sha_of(path):
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()

def env_record():
    # 판이 하나라도 다르면 같은 설정에서도 출력이 달라질 수 있다(스펙 6-2절). torch 는 코랩에 깔린 것을 적기만 한다.
    env = {'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'python': platform.python_version(),
           'transformers': transformers.__version__, 'bitsandbytes': bitsandbytes.__version__,
           'peft': peft.__version__, 'accelerate': accelerate.__version__}
    wrong = {k: (v, env[k]) for k, v in PINS.items() if env[k] != v}
    assert not wrong, f'고정한 판과 다르게 깔렸습니다: {wrong}. 런타임을 다시 시작하고 첫 칸부터 다시 실행하세요.'
    return env

def check_env(saved_path):
    # 이어 할 때는 처음 세션과 판이 같아야 한다. 다르면 이어 붙인 결과가 한 환경의 결과가 아니게 된다.
    if os.path.exists(saved_path):
        old = json.load(open(saved_path))
        diff = {k: (old.get(k), ENV.get(k)) for k in set(old) | set(ENV) if old.get(k) != ENV.get(k)}
        assert not diff, f'처음 세션과 환경이 다릅니다: {diff}. 멈추고 이 줄을 그대로 알려 주세요.'
    else:
        json.dump(ENV, open(saved_path, 'w'), indent=2)

QUANT = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                           bnb_4bit_compute_dtype=torch.float16)  # v2 와 같다. T4 는 bf16 을 지원하지 않는다

def load_base():
    tok = AutoTokenizer.from_pretrained(M3_REPO, revision=M3_REVISION, token=HF_TOKEN)
    # dtype=float16: 4비트로 바뀌지 않는 부분(임베딩, lm_head, 정규화층)의 정밀도. v2 노트북과 같다(스펙 6-1절).
    model = AutoModelForCausalLM.from_pretrained(M3_REPO, revision=M3_REVISION, quantization_config=QUANT,
                                                 dtype=torch.float16, device_map='auto', token=HF_TOKEN)
    return tok, model

assert SHEET in ('new', 'frozen') and MODEL in ('M3pre', 'M3ft') and RUN in (1, 2, 3)
# 학습 전 M3 에 해시가 적혀 있으면 어느 쪽을 돌리려던 것인지 알 수 없다. 결과 줄의 adapter_sha256 도 거짓이 된다.
assert (MODEL == 'M3pre') == (ADAPTER_SHA256 == ''), 'M3pre 이면 ADAPTER_SHA256 을 비우고, M3ft 이면 채웁니다.'
EXPECTED = {'new': '', 'frozen': 'bec2b58e943e3b2c296dec8b664dc78ed2548f1b6d6e67062e3a7684fd535fe2'}
MAX_NEW_TOKENS = 512
SMOKE_MESSAGES = [{"role": "system", "content": "You are a fair judge assistant tasked with providing clear, objective feedback based on specific criteria, ensuring each assessment reflects the absolute standards set for performance."}, {"role": "user", "content": "###Task Description:\nAn instruction (might include an Input inside it), a response to evaluate, and a score rubric representing a evaluation criteria are given.\n1. Write a detailed feedback that assess the quality of the response strictly based on the given score rubric, not evaluating in general.\n2. After writing a feedback, write a score that is an integer between 1 and 5. You should refer to the score rubric.\n3. The output format should look as follows: \"Feedback: (write a feedback for criteria) [RESULT] (an integer number between 1 and 5)\"\n4. Please do not generate any other opening, closing, and explanations.\n\n###The instruction to evaluate:\n아래 근거만 사용해서 질문에 답하세요.\n\n<근거>\n[근거 1] (출처: 가짜.md)\n정규직 직원의 연차는 15일이다.\n</근거>\n\n<질문>\n연차는 며칠인가요?\n\n###Response to evaluate:\n연차는 15일이고, 회사 창립기념일에는 하루 더 쉽니다.\n\n###Score Rubrics:\n[충실성: 답변의 모든 주장이 근거에 실제로 적혀 있는가. 답변이 세상의 상식에 맞는지는 보지 않는다. 사실이더라도 근거에 없으면 점수를 낮춘다.]\nScore 1: 답변이 근거와 관계없는 내용을 지어냈거나, 근거와 반대되는 내용을 말한다\nScore 2: 답변의 주장 대부분이 근거에 없다\nScore 3: 근거에 있는 주장과 근거에 없는 주장이 비슷한 정도로 섞여 있다\nScore 4: 답변의 핵심 답은 근거와 일치한다. 다만 근거에 없는 사소한 주장이 하나 있다\nScore 5: 답변의 모든 주장이 근거에 적혀 있다\n\n###Feedback: "}]
CASES = f'{DIR}/cases_{SHEET}.jsonl'
assert EXPECTED[SHEET], '동결 커밋 전의 노트북입니다. 계획 E11 뒤의 노트북을 쓰세요.'
sha = sha_of(CASES)
assert sha == EXPECTED[SHEET], f'시험지 해시가 다릅니다: {sha}'
CASE_ROWS = [json.loads(l) for l in open(CASES, encoding='utf-8') if l.strip()]
OUT = f'{DIR}/results/{SHEET}'
os.makedirs(OUT, exist_ok=True)
PATH = f'{OUT}/{MODEL}_run{RUN}.jsonl'
ENV = env_record()
check_env(PATH.replace('.jsonl', '.env.json'))   # 첫 세션이면 적고, 이어 하는 세션이면 처음과 비교한다
print(len(CASE_ROWS), '문항', ENV)


In [ ]:
tok, model = load_base()
if MODEL == 'M3ft':
    from peft import PeftModel
    assert ADAPTER_SHA256 and sha_of(f'{DIR}/adapter/adapter_model.safetensors') == ADAPTER_SHA256, '어댑터 해시가 다릅니다'
    model = PeftModel.from_pretrained(model, f'{DIR}/adapter')   # 어댑터를 바탕 모델 위에 얹는다(합치지 않는다)
model.eval()
# 생성 설정은 v2 노트북과 같게 처음부터 만든다. 저장소 기본값의 repetition_penalty 1.05 가 섞이지 않게 한다.
eos = model.generation_config.eos_token_id
pad = tok.pad_token_id if tok.pad_token_id is not None else (eos[0] if isinstance(eos, list) else eos)
gen = GenerationConfig(max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.0,
                       eos_token_id=eos, pad_token_id=pad)

def generate(messages):
    text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    enc = tok(text, return_tensors='pt', add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, generation_config=gen, use_model_defaults=False)
    new = out[0, enc['input_ids'].shape[1]:]
    finish = 'length' if new.shape[0] >= MAX_NEW_TOKENS else 'stop'
    return tok.decode(new, skip_special_tokens=True), finish

text, finish = generate(SMOKE_MESSAGES)
assert text.strip(), '가짜 문항의 출력이 비었다. 멈추고 알린다'
json.dump({'model': MODEL, 'output': text, 'finish': finish, 'generation': gen.to_dict()},
          open(f'{OUT}/{MODEL}_run{RUN}_smoke.json', 'w'), ensure_ascii=False, indent=2)
def resume_lines():
    """이미 쓴 줄. 마지막 줄이 쓰다가 끊겨 JSON 이 아니면 버리고 파일을 다시 쓴다. 버린 것은 기록한다."""
    if not os.path.exists(PATH):
        return []
    raw = [l for l in open(PATH, encoding='utf-8').read().split('\n') if l.strip()]
    good = []
    for i, l in enumerate(raw):
        try:
            good.append(json.loads(l))
        except json.JSONDecodeError:
            assert i == len(raw) - 1, f'{PATH} 의 {i + 1}번째 줄이 깨졌습니다. 마지막 줄이 아니라 멈춥니다.'
            open(PATH, 'w', encoding='utf-8').write(''.join(json.dumps(g, ensure_ascii=False) + '\n' for g in good))
            open(f'{OUT}/{MODEL}_run{RUN}_resume.log', 'a').write(json.dumps({'dropped_partial_line': l[:200], 'at': time.time()}) + '\n')
            print('반쯤 쓴 마지막 줄을 버리고 이어 한다')
    return good

done = {g['case_id'] for g in resume_lines()}
for i, row in enumerate(CASE_ROWS):
    if row['case_id'] in done:
        continue
    text, finish = generate(row['messages'])
    line = {'case_id': row['case_id'], 'model': MODEL, 'repo': M3_REPO, 'revision': M3_REVISION, 'run': RUN,
            'output': text, 'finish': finish, 'cases_sha256': sha, 'domain': row['domain'],
            'adapter_sha256': ADAPTER_SHA256 or None}
    with open(PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(line, ensure_ascii=False) + '\n')
    print(i + 1, '/', len(CASE_ROWS), finish)   # 점수는 찍지 않는다. 학습이 끝날 때까지 결과를 보지 않는다


In [ ]:
name = f'alldap_judgeft_{SHEET}_{MODEL}_run{RUN}'
os.makedirs(f'/content/{name}', exist_ok=True)
for p in glob.glob(f'{OUT}/{MODEL}_run{RUN}*'):
    shutil.copy(p, f'/content/{name}/')
shutil.make_archive(f'/content/{name}', 'zip', f'/content/{name}')
files.download(f'/content/{name}.zip')
